# **NOTEBOOK 7: Streamlit App**

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import TargetEncoder
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error, median_absolute_error
from sklearn.metrics import mean_absolute_percentage_error
from matplotlib import pyplot as plt
import seaborn as sns
import geopandas as gpd
from geopandas import GeoDataFrame
from shapely.geometry import Point
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

## Load Dataset

In [27]:
df = pd.read_csv('preprocessed_data.csv')
df.head()

,ViewYN,PoolPrivateYN,CloseDate,ClosePrice,Latitude,Longitude,LivingArea,AttachedGarageYN,ParkingTotal,BathroomsTotalInteger,...,Windsor Unified,Winters Joint Unified,Wiseburn Unified,Woodlake Unified,Woodland Joint Unified,Yosemite Unified,Yuba City Unified,Yucaipa-Calimesa Joint Unified,PropertyAge,BedBathRatio
0,1,1,2022-01-04,2499999.0,33.147270,-117.340604,2645.0,1,2.0,4.0,...,0,0,0,0,0,0,0,0,6.0,1.0
1,1,1,2022-01-10,640000.0,34.235979,-117.202269,2070.0,1,1.0,3.0,...,0,0,0,0,0,0,0,0,15.0,1.0
2,0,1,2022-03-23,438000.0,34.114130,-117.442493,1174.0,1,2.0,2.0,...,0,0,0,0,0,0,0,0,62.0,1.5
3,1,1,2022-01-10,615000.0,33.783767,-116.447280,1996.0,0,4.0,2.0,...,0,0,0,0,0,0,0,0,17.0,1.0
4,1,1,2022-01-19,399990.0,39.767729,-121.586400,1422.0,1,2.0,2.0,...,0,0,0,0,0,0,0,0,1.0,1.5


## Train/Test Split

In [3]:
def train_test_split(df, train_months):

    df = df.copy()

    # ensure datetime
    df['CloseDate'] = pd.to_datetime(df['CloseDate'])

    # get monthly periods
    df['CloseMonth'] = df['CloseDate'].dt.to_period('M')

    # most recent month becomes test
    latest_month = df['CloseMonth'].max()

    test_df = df[df['CloseMonth'] == latest_month]

    # X months immediately before test month
    train_start = latest_month - train_months

    train_df = df[(df['CloseMonth'] < latest_month) & (df['CloseMonth'] >= train_start)]

    # remove date columns
    train_df = train_df.drop(columns=['CloseDate', 'CloseMonth'])

    test_df = test_df.drop(columns=['CloseDate', 'CloseMonth'])

    # separate target
    X_train = train_df.drop(columns=['ClosePrice'])
    y_train = train_df['ClosePrice']

    X_test = test_df.drop(columns=['ClosePrice'])
    y_test = test_df['ClosePrice']

    return X_train, X_test, y_train, y_test

## Filter Outliers/Extremes

In [4]:
def data_filter(X_train,
           X_test,
           y_train,
           y_test,
           lower_threshold=0.005,
           upper_threshold=0.995):
    
    # compute ClosePrice thresholds from training set only
    lower = y_train.quantile(lower_threshold)
    upper = y_train.quantile(upper_threshold)
    
    # apply thresholds to train and test
    train_filter = y_train.between(lower, upper)
    test_filter = y_test.between(lower, upper)
    
    X_train = X_train[train_filter]
    y_train = y_train[train_filter]
    X_test = X_test[test_filter]
    y_test = y_test[test_filter]

    return X_train, X_test, y_train, y_test

## Evaluation Metrics

In [10]:
# define MdAPE
def mdape(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    mask = y_true != 0
    ape = np.abs((y_true[mask] - y_pred[mask]) / y_true[mask]) * 100
    return np.median(ape)

# define metrics function
def metrics(y_true, y_pred):
      return {
          'R2': f"{r2_score(y_true, y_pred):.4f}",
          'MAE': f"${mean_absolute_error(y_true, y_pred):.2f}",
          'RMSE': f"${root_mean_squared_error(y_true, y_pred):.2f}",
          'MedianAbsoluteError': f"${median_absolute_error(y_true, y_pred):.2f}",
          'MAPE': f"{mean_absolute_percentage_error(y_true, y_pred) * 100:.2f}%",
          'MDAPE': f"{mdape(y_true, y_pred):.2f}%"
      }

## LightGBM Model

In [6]:
# train/test split
X = 30
X_train, X_test, y_train, y_test = train_test_split(df, X)

# filter extremes
X_train, X_test, y_train, y_test = data_filter(X_train,
                                               X_test,
                                               y_train,
                                               y_test,
                                               lower_threshold=0.005,
                                               upper_threshold=0.995)

In [7]:
X_train.head()

,ViewYN,PoolPrivateYN,Latitude,Longitude,LivingArea,AttachedGarageYN,ParkingTotal,BathroomsTotalInteger,BedroomsTotal,FireplaceYN,...,Windsor Unified,Winters Joint Unified,Wiseburn Unified,Woodlake Unified,Woodland Joint Unified,Yosemite Unified,Yuba City Unified,Yucaipa-Calimesa Joint Unified,PropertyAge,BedBathRatio
3748,0,1,33.941807,-118.239928,1350.0,0,2.0,2.0,4.0,0,...,0,0,0,0,0,0,0,0,49.0,2.000000
4957,0,1,32.659210,-117.097531,1974.0,1,4.0,4.0,4.0,0,...,0,0,0,0,0,0,0,0,0.0,1.000000
4958,0,1,32.659251,-117.097312,1974.0,1,2.0,4.0,4.0,0,...,0,0,0,0,0,0,0,0,0.0,1.000000
4985,1,1,34.534363,-117.960529,2400.0,1,2.0,3.0,4.0,0,...,0,0,0,0,0,0,0,0,16.0,1.333333
5483,1,1,33.524038,-117.023755,5256.0,1,5.0,7.0,6.0,1,...,0,0,0,0,0,0,0,0,1.0,0.857143


In [11]:
# LightGBM
lgbm = LGBMRegressor(random_state=123, max_depth=9, learning_rate=0.2, n_estimators=1000, verbose=-1)
lgbm.fit(X_train, y_train)
lgbm_pred = lgbm.predict(X_test)
lgbm_metrics = metrics(y_test, lgbm_pred)

In [14]:
lgbm_metrics

{'R2': '0.9047',
 'MAE': '$160612.56',
 'RMSE': '$300871.88',
 'MedianAbsoluteError': '$79023.08',
 'MAPE': '12.31%',
 'MDAPE': '8.76%'}

## Retrain LightGBM Model with Simplified Data

In [30]:
features = [
    'LivingArea',
    'BedroomsTotal',
    'BathroomsTotalInteger',
    'LotSizeSquareFeet',
    'ClosePrice',
    'CloseDate'
]

simple_df = df[features]
simple_df.head()

,LivingArea,BedroomsTotal,BathroomsTotalInteger,LotSizeSquareFeet,ClosePrice,CloseDate
0,2645.0,4.0,4.0,13376.0,2499999.0,2022-01-04
1,2070.0,3.0,3.0,3397.0,640000.0,2022-01-10
2,1174.0,3.0,2.0,9900.0,438000.0,2022-03-23
3,1996.0,2.0,2.0,6098.0,615000.0,2022-01-10
4,1422.0,3.0,2.0,12197.0,399990.0,2022-01-19


In [31]:
# train/test split
X = 30
X_simple_train, X_simple_test, y_simple_train, y_simple_test = train_test_split(simple_df, X)

# filter extremes
X_simple_train, X_simple_test, y_simple_train, y_simple_test = data_filter(X_simple_train,
                                               X_simple_test,
                                               y_simple_train,
                                               y_simple_test,
                                               lower_threshold=0.005,
                                               upper_threshold=0.995)

In [34]:
X_simple_train.head()

,LivingArea,BedroomsTotal,BathroomsTotalInteger,LotSizeSquareFeet
3748,1350.0,4.0,2.0,6308.0
4957,1974.0,4.0,4.0,7245.0
4958,1974.0,4.0,4.0,7245.0
4985,2400.0,4.0,3.0,39587.0
5483,5256.0,6.0,7.0,87120.0


In [35]:
# simplified LightGBM
simple_lgbm = LGBMRegressor(random_state=123, max_depth=9, learning_rate=0.2, n_estimators=1000, verbose=-1)
simple_lgbm.fit(X_simple_train, y_simple_train)
simple_lgbm_pred = simple_lgbm.predict(X_simple_test)
simple_lgbm_metrics = metrics(y_simple_test, simple_lgbm_pred)

In [36]:
simple_lgbm_metrics

{'R2': '0.4434',
 'MAE': '$470536.03',
 'RMSE': '$727112.68',
 'MedianAbsoluteError': '$309220.88',
 'MAPE': '45.46%',
 'MDAPE': '31.06%'}

## Import Model to Joblib

In [25]:
import joblib

In [37]:
joblib.dump(simple_lgbm, 'lgbm_model.pkl')

['lgbm_model.pkl']